# Support Vector Regression

Standard tabular ML workflow for LD50 regression.

In [4]:
from pathlib import Path
import sys
import warnings

import numpy as np
from sklearn.svm import SVR as SklearnSVR

MODELING_DIR = Path("../modeling").resolve()
if str(MODELING_DIR) not in sys.path:
    sys.path.append(str(MODELING_DIR))

from evaluation import save_ml_artifacts
from evaluation import evaluate_model
from preprocessing import load_and_preprocess_tabular
from storage import artifact_paths


ModuleNotFoundError: No module named 'evaluation'

In [ ]:
MODEL_NAME = "svm"
BASE_DIR = Path.cwd()
RANDOM_STATE = 42
MAX_TRAIN_ROWS = 50_000

X_train, X_val, X_test, y_train, y_val, y_test, feature_names, preprocessor = load_and_preprocess_tabular()
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)
paths = artifact_paths(BASE_DIR, MODEL_NAME, model_suffix=".pkl", importance=True)


In [ ]:
try:
    from cuml.svm import SVR as CumlSVR

    model = CumlSVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale")
    use_gpu = True
except Exception:
    warnings.warn("cuML SVR not available; using sklearn SVR on CPU.")
    model = SklearnSVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale")
    use_gpu = False

if X_train.shape[0] > MAX_TRAIN_ROWS:
    sample_idx = np.random.RandomState(RANDOM_STATE).choice(
        X_train.shape[0],
        size=MAX_TRAIN_ROWS,
        replace=False,
    )
    X_fit = X_train[sample_idx]
    y_fit = y_train[sample_idx]
else:
    X_fit = X_train
    y_fit = y_train

model.fit(X_fit, y_fit)
predictions = model.predict(X_test)


In [ ]:
metrics = save_ml_artifacts(
    model_name="SVM",
    model=model,
    paths=paths,
    X_test=X_test,
    y_test=y_test,
    predictions=predictions,
    feature_names=feature_names,
    plot_color="green",
    use_permutation_importance=True,
)
metrics
